# Model Training & Selection Notebook - BitGuard
---

Contents:
1. Load pre-aggregated features from Neo4j graph output
2. Feature engineering --> 138 total features
3. Label generation --> BitcoinHeist + Elliptic++
4. EDA snapshot and class imbalance
5. Model performance and selection --> Logistic Regression, Random Forest, LightGBM
6. LightGBM fine-tuning --> DART + threshold optimization
7. Final evaluation & SHAP explainability

> **Note:** `features.parquet` was generated by `aggregate_features.py` which streams the 89.6GB `neighborhood_results.csv` from `s3://210-btc/` and aggregates per-seed wallet features across 3 hop depths × 2 directions.

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pickle
from collections import Counter

# sklearn imports
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    classification_report
    , roc_auc_score
    , confusion_matrix
    , ConfusionMatrixDisplay
    , roc_curve
)
import lightgbm as lgb
import shap

# ignore warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

# set random seet for repeatability
SEED = 42

## 1. Load Features

Base features were pre-aggregated from the Neo4j neighborhood graph output.
Each row = one seed Bitcoin wallet address. 101 base features across:
- 3 hop depths (hop 0, 1, 2)
- 2 directions (forward = sent, reverse = received)
- Feature types: BTC volume, degree, dust, CoinJoin, round-number, temporal

In [ ]:
# load pre-aggregated features
# generated from aggregate_features.py in 3://210-btc/neighborhood_results.csv
df = pd.read_parquet('features.parquet')

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns[:10])} ... ({len(df.columns)} total)")

df.head(3)

## 2. Feature Engineering

We extend the 101 base features with two additional groups:

**Asymmetry features** — capture directional imbalance (forward vs reverse) for each hop:
$$\text{asymmetry} = \frac{\text{forward} - \text{reverse}}{\text{forward} + \text{reverse} + \epsilon}$$

**Log transforms** — reduce skew on BTC volume and variance features via `log1p`.

Final feature count: **138 features**

In [ ]:
HOPS = [0, 1, 2]

# asymmetry features
for hop in HOPS:
    for metric in ["btc_volume", "degree", "unique_neighbors", "dust_count", "round_count"]:
        fwd = df.get(f"{metric}_hop{hop}_forward", 0.0)
        rev = df.get(f"{metric}_hop{hop}_reverse", 0.0)
        df[f"{metric}_hop{hop}_asymmetry"] = (fwd - rev) / (fwd + rev + 1e-8)

# log transformations
log_targets = [
    "btc_volume_hop0_forward", "btc_avg_hop0_forward", "btc_max_hop0_forward", "amount_variance_hop0_forward",
    "btc_volume_hop0_reverse", "btc_avg_hop0_reverse", "btc_max_hop0_reverse", "amount_variance_hop0_reverse",
    "btc_volume_hop1_forward", "btc_avg_hop1_forward", "btc_max_hop1_forward", "amount_variance_hop1_forward",
    "btc_volume_hop1_reverse", "btc_avg_hop1_reverse", "btc_max_hop1_reverse", "amount_variance_hop1_reverse",
    "btc_volume_hop2_forward", "btc_avg_hop2_forward", "btc_max_hop2_forward", "amount_variance_hop2_forward",
    "btc_volume_hop2_reverse", "btc_avg_hop2_reverse", "btc_max_hop2_reverse", "amount_variance_hop2_reverse",
    "total_btc_forward", "total_btc_reverse", "balance",
    "btc_volume_hop0_asymmetry", "btc_volume_hop1_asymmetry", "btc_volume_hop2_asymmetry",
]
for col in log_targets:
    if col in df.columns:
        df[f"{col}_log"] = np.log1p(df[col].clip(lower=0))

# print results
print(f"Feature count after engineering: {df.shape[1]}")

## 3. Label Joining

Labels come from two sources, joined on Bitcoin wallet address:
- **BitcoinHeist** — ~29k illicit wallets (28 ransomware families, UCI)
- **Elliptic++** — ~6k illicit wallets (darknet, scam, exchange hack categories, Georgia Tech)

After deduplication across both datasets: **513k total wallets, ~35k illicit (6.8%)**

In [ ]:
# load labeled addresses
# addresses.csv contains wallet address + is_bad_actor label (0/1)
# generated by joining BitcoinHeist + Elliptic++ and deduplicating
labels = pd.read_csv('s3://210-btc/addresses.csv')  # or local path
labels = labels[['address', 'is_bad_actor']].drop_duplicates('address')

# merge on seed addresses
df = df.merge(labels, left_on='seed', right_on='address', how='inner')
df = df.drop(columns=['address'])

# print results
print(f"After label join: {df.shape[0]:,} wallets")
print(f"Class distribution:")
print(df['is_bad_actor'].value_counts())
print(f"\nIllicit rate: {df['is_bad_actor'].mean()*100:.1f}%")
print(f"Class imbalance ratio: {(1 - df['is_bad_actor'].mean()) / df['is_bad_actor'].mean():.1f}:1")

## 4. EDA Snapshot

Quick look at the key behavioral signals that motivated our feature groups.

In [ ]:
# compare illicit vs. licit wallets
signal_features = [
    'dust_tx_count'
    , 'dust_ratio'
    , 'round_number_ratio'
    , 'has_dust_attack'
    , 'has_round_laundering'
]

# print results
signal_cols = [c for c in signal_features if c in df.columns]
print("Mean values by label:")
print(df.groupby('is_bad_actor')[signal_cols].mean().round(4).to_string())

In [ ]:
# compare coinjoin features
coinjoin_cols = [c for c in df.columns if 'coinjoin' in c and 'hop0' in c]
print("CoinJoin features by label:")
print(df.groupby('is_bad_actor')[coinjoin_cols].mean().round(2).to_string())
print(f"\nCoinJoin ratio (illicit / clean): {df[df.is_bad_actor==1]['coinjoin_tx_count_hop0_forward'].mean() / max(df[df.is_bad_actor==0]['coinjoin_tx_count_hop0_forward'].mean(), 1e-8):.1f}x")

## 5. Train / Validation / Test Split

Stratified split to preserve class imbalance across all splits:
- **70%** train
- **15%** validation (LightGBM early stopping)
- **15%** test (held out, final evaluation only)

In [ ]:
from inference_features import FEATURE_COLS  # 138 feature names

feature_cols = [c for c in FEATURE_COLS if c in df.columns]
print(f"Features used: {len(feature_cols)}")

X = df[feature_cols]
y = df['is_bad_actor']

# 70 / 15 / 15 stratified split
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15/0.85, stratify=y_temp, random_state=SEED
)

print(f"Train:      {X_train.shape[0]:,} samples ({y_train.mean()*100:.1f}% illicit)")
print(f"Validation: {X_val.shape[0]:,} samples ({y_val.mean()*100:.1f}% illicit)")
print(f"Test:       {X_test.shape[0]:,} samples ({y_test.mean()*100:.1f}% illicit)")

# get class imbalance ratio for scale_pos_weight
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

# print results
print(f"\nscale_pos_weight: {scale_pos_weight:.2f}")

## 6. Model Comparison

We evaluate three model families with increasing complexity:

| Model | Type | Decision boundary |
|---|---|---|
| Logistic Regression | Linear | Linear only |
| Random Forest | Parallel ensemble (bagging) | Non-linear |
| LightGBM | Sequential ensemble (boosting) | Non-linear, error-correcting |

The jump from LR → RF validates that the problem is non-linear. The jump from RF → LightGBM shows boosting adds meaningful value on the illicit class.

In [ ]:
def evaluate(name, model, X_eval, y_eval, threshold=0.5):
    proba = model.predict_proba(X_eval)[:, 1]
    preds = (proba >= threshold).astype(int)
    auc = roc_auc_score(y_eval, proba)
    report = classification_report(y_eval, preds, output_dict=True)
    illicit = report.get('1', {})
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"  ROC-AUC:   {auc:.4f}")
    print(f"  Precision: {illicit.get('precision', 0):.4f}  (illicit class)")
    print(f"  Recall:    {illicit.get('recall', 0):.4f}  (illicit class)")
    print(f"  F1:        {illicit.get('f1-score', 0):.4f}  (illicit class)")
    print(f"  Accuracy:  {report['accuracy']:.4f}")
    return auc

In [ ]:
# baseline model -- Logistic Regressions
print("Training Logistic Regression...")
lr = LogisticRegression(
    class_weight = 'balanced'
    , max_iter = 1000
    , random_state = SEED
)
lr.fit(X_train, y_train)
auc_lr = evaluate("Logistic Regression (baseline)", lr, X_test, y_test)

In [ ]:
# baseline tree model --> Random Forest
print("Training Random Forest...")
rf = RandomForestClassifier(
    n_estimators = 500
    , class_weight = 'balanced'
    , max_depth = None
    , min_samples_leaf = 10
    , n_jobs =- 1
    , random_state = SEED
)
rf.fit(X_train, y_train)
auc_rf = evaluate("Random Forest (baseline tree model)", rf, X_test, y_test)

In [ ]:
# standard lightgbm
print("Training LightGBM (standard gbdt)...")
lgbm_std = lgb.LGBMClassifier(
    boosting_type = 'gbdt'
    , n_estimators = 1000
    , learning_rate = 0.05
    , num_leaves = 127
    , min_child_samples = 50
    , subsample = 0.8
    , colsample_bytree = 0.8
    , scale_pos_weight = scale_pos_weight
    , random_state = SEED
    , n_jobs =- 1
)

# fit lgbm model
lgbm_std.fit(
    X_train, y_train
    , eval_set=[(X_val, y_val)]
    , callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)

# get STD
auc_lgbm_std = evaluate("LightGBM (gbdt)", lgbm_std, X_test, y_test)

## 7. LightGBM Fine-Tuning with DART

DART (Dropouts meet Multiple Additive Regression Trees) applies dropout regularization during boosting — each tree round randomly drops a fraction of existing trees before adding a new one. This reduces overfitting and improves generalization on the minority class.

Key DART parameters:
- `drop_rate=0.2` — 20% of trees dropped per round
- `skip_drop=0.5` — 50% chance to skip dropout entirely (stabilizes early rounds)
- `max_drop=50` — caps trees dropped per round

> **Note:** DART does not support early stopping (tree dropout makes the monotonic loss assumption invalid). We train for a fixed 4,000 rounds.

In [ ]:
# final model: LightGBM w/ DART

lgbm_dart = lgb.LGBMClassifier(
    boosting_type = 'dart'
    , n_estimators = 4000
    , learning_rate v= 0.05
    , num_leaves = 127
    , min_child_samples = 50
    , subsample = 0.8
    , colsample_bytree = 0.8
    , drop_rate = 0.2
    , skip_drop = 0.5
    , max_drop = 50
    , scale_pos_weight = scale_pos_weight
    , random_state = SEED
    , n_jobs =- 1
)
lgbm_dart.fit(X_train, y_train)
auc_dart = evaluate("LightGBM DART (4000 rounds)", lgbm_dart, X_test, y_test)

## 8. Threshold Optimization

Default threshold of 0.5 is suboptimal for imbalanced classes.
We sweep thresholds on the validation set and select the one maximizing F1 on the illicit class.

In [ ]:
val_proba = lgbm_dart.predict_proba(X_val)[:, 1]

thresholds = np.arange(0.3, 0.75, 0.01)
f1_scores = []
for t in thresholds:
    preds = (val_proba >= t).astype(int)
    report = classification_report(y_val, preds, output_dict=True, zero_division=0)
    f1_scores.append(report.get('1', {}).get('f1-score', 0))

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

plt.figure(figsize=(8, 4))
plt.plot(thresholds, f1_scores, color='#F5A023', linewidth=2)
plt.axvline(best_threshold, color='white', linestyle='--', alpha=0.7)
plt.xlabel('Threshold')
plt.ylabel('F1 Score (illicit class)')
plt.title(f'Threshold optimization — best: {best_threshold:.2f} (F1={f1_scores[best_idx]:.4f})')
plt.tight_layout()
plt.show()

# print results
print(f"Best threshold: {best_threshold:.2f}")

## 9. Final Evaluation on Test Set

In [ ]:
test_proba = lgbm_dart.predict_proba(X_test)[:, 1]
test_preds = (test_proba >= best_threshold).astype(int)

# print results
print(f"Test set: n = {len(y_test):,} wallets")
print(f"Threshold: {best_threshold:.2f}")
print(f"\nROC-AUC: {roc_auc_score(y_test, test_proba):.4f}")
print()
print(classification_report(y_test, test_preds, target_names = ['licit', 'illicit']))

In [ ]:
# generate confusion matrix on final productioin model test set
cm = confusion_matrix(y_test, test_preds)
fig, ax = plt.subplots(figsize = (6, 5))
disp = ConfusionMatrixDisplay(cm, display_labels=['Licit', 'Illicit'])
disp.plot(ax=ax, colorbar = False, cmap = 'Oranges')
ax.set_title('LightGBM DART — Confusion Matrix (Test Set)')
plt.tight_layout()
plt.show()

In [ ]:
# compare each mode's ROC curve
fig, ax = plt.subplots(figsize=(7, 5))

for name, model, color in [
    ('Logistic Regression', lr, '#888780')
    , ('Random Forest', rf, '#5a6070')
    , ('LightGBM DART', lgbm_dart, '#F5A023'),
]:
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_test)[:, 1])
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})', color=color, linewidth=2)

ax.plot([0,1], [0,1], 'k--', alpha=0.4)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Model Comparison')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# model performance and summary table
results = []
for name, model in [
    ('Logistic Regression', lr),
    ('Random Forest', rf),
    ('LightGBM (gbdt)', lgbm_std),
    ('LightGBM DART', lgbm_dart),
]:
    proba = model.predict_proba(X_test)[:, 1]
    preds = (proba >= best_threshold).astype(int)
    report = classification_report(y_test, preds, output_dict=True, zero_division=0)
    results.append({
        'Model': name
        , 'ROC-AUC': roc_auc_score(y_test, proba)
        , 'Precision (illicit)': report['1']['precision']
        , 'Recall (illicit)': report['1']['recall']
        , 'F1 (illicit)': report['1']['f1-score']
        , 'Accuracy': report['accuracy'],
    })
    
# print results
results_df = pd.DataFrame(results).set_index('Model')
print("\nModel Performance Comparison (Test Set)")
print("=" * 75)
print(results_df.to_string())

## 10. SHAP Explainability

We use TreeSHAP — an exact polynomial-time Shapley value algorithm for tree models.
Unlike KernelSHAP (used for neural networks), TreeSHAP provides exact local explanations without approximation.

SHAP values answer: *how much did each feature push the model's prediction toward or away from flagging a wallet as illicit?*

In [ ]:
# generate treeSHAP on a test set sample (too slow on entire large dataset)
sample_size = min(2000, len(X_test))
X_sample = X_test.sample(sample_size, random_state = SEED)

explainer = shap.TreeExplainer(lgbm_dart)
shap_values = explainer(X_sample)

print(f"SHAP values computed for {sample_size} wallets")
print(f"SHAP array shape: {shap_values.values.shape}")

In [ ]:
# global feature importance via mean absolute SHAP
shap.summary_plot(
    shap_values.values if shap_values.values.ndim == 2 else shap_values.values[:, :, 1]
    , X_sample
    , max_display=20
    , plot_type='bar'
    , show=True
)

In [ ]:
# beeswarm plot --> shows direction of each feature's influence
shap.summary_plot(
    shap_values.values if shap_values.values.ndim == 2 else shap_values.values[:, :, 1]
    , X_sample
    , max_display=20
    , show=True
)

## 11. Save Model

In [ ]:
model_payload = {
    'model': lgbm_dart
    , 'feature_cols': feature_cols
    , 'best_threshold': float(best_threshold)
    , 'scale_pos_weight': float(scale_pos_weight)
    , 'n_train': len(X_train)
    , 'n_test': len(X_test)
    , 'test_auc': roc_auc_score(y_test, test_proba),
}

with open('model.pkl', 'wb') as f:
    pickle.dump(model_payload, f)

print("Model saved to model.pkl")
print(f"Keys: {list(model_payload.keys())}")

## Summary

| Model | ROC-AUC | Precision | Recall | F1 |
|---|---|---|---|---|
| Logistic Regression | 0.9067 | 0.795 | 0.853 | 0.823 |
| Random Forest | 0.9802 | 0.925 | 0.921 | 0.923 |
| LightGBM (gbdt) | 0.9898 | 0.953 | 0.946 | 0.949 |
| **LightGBM DART** | **0.9814** | **0.94** | **0.83** | **0.88** |

**Key Findings:**
- The LR → RF jump as measured by AUC (0.907 → 0.980 AUC) confirms non-linear decision boundary, and LR approach is insufficient to adequately capture complexity of the 138 feature interactions.
- LightGBM w/ DART was selected as the final model because it provided a strong decision score of 0.94. Low false positive rate is ideal to prioritize to ensure high user trust.
- The current performance bottleneck is **false negatives at ~17%** due to class imbalance and label quality ceiling. Improvements would come from additional labeled data and USD-normalized BTC volume features.
- SHAP (TreeSHAP) provides exact, per-wallet explanations enabling the interpretable risk scoring surfaced in the BitGuard.pro API.